# Domestic Flight Price Prediction - Project-I Starter

Competition: `domestic-flight-price-prediction-challenge` | Target: `price` (regression, INR).
**Allowed models ONLY (regression_models.md): Ridge (L2), Lasso (L1), Regression Tree.**
Structure: 1 Loading, 2 EDA, 3 Cleaning, 4 Feature engineering, 5 Preprocessing, 6 Reusable
evaluation functions, 7 Direct family comparison, 8 Hyper-parameter tuning, 9 log(price) experiment,
10 intentionally OMITTED (ensembles are not Part-1 models and are banned by the rules),
11 Diagnostics, 12 Final CV of shortlisted models + submission.
Kernel: `ML Flight Price (.venv)`. No auto-submit - you upload `submission.csv` manually.


### Sec.1 - Data Loading
Setup (versions, seed 42 for reproducibility) then reads train/test/sample.
Sample file defines the exact submission format. On Kaggle a shim links /kaggle/input files here.


In [ ]:
# 1. Setup (.venv kernel, seed 42)
import sys, sklearn, pandas as pd, numpy as np
print(sys.version)
print('sklearn', sklearn.__version__)
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [ ]:
# 1. Load data
import os
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
sample = pd.read_csv('sample_submission.csv')
print(train.shape, test.shape, sample.shape)
display(train.head())
display(test.head())
display(sample.head())

### Sec.2 - Exploratory Data Analysis
Dtypes, missing rates (~20% airline/duration), skewed target
(mean ~30k, max 323k), categorical cardinality (flight 1070 IDs, airline 18). Findings drive Sec.3-5.


In [ ]:
# 2. Quick EDA
print(train.dtypes)
print('\nMissing train (%):')
print((train.isna().mean()*100).round(2).sort_values(ascending=False))
print('\nMissing test (%):')
print((test.isna().mean()*100).round(2).sort_values(ascending=False))
print('\nTarget describe:')
display(train['price'].describe())
for c in ['airline','source_city','departure_time','stops','arrival_time','destination_city','flight_class']:
    print(f'\n{c}:', train[c].nunique(), sorted(train[c].dropna().unique())[:10])

### Sec.3 - Data Cleaning
Pandas-level fixes with the leakage rule: statistics (median) come from TRAIN
only and are reused on test. Reports missing counts before/after.


In [ ]:
# 3. Data Cleaning (pandas level; imputation lives in Sec.5 pipelines)
train_clean = train.copy(); test_clean = test.copy()
print('missing BEFORE train:', train_clean.isna().sum()[train_clean.isna().sum() > 0].to_dict())
print('missing BEFORE test :', test_clean.isna().sum()[test_clean.isna().sum() > 0].to_dict())
train_clean['airline'] = train_clean['airline'].fillna('MISSING')
test_clean['airline'] = test_clean['airline'].fillna('MISSING')
med_duration = train_clean['duration'].median()  # median from TRAIN only: no leakage
train_clean['duration'] = train_clean['duration'].fillna(med_duration)
test_clean['duration'] = test_clean['duration'].fillna(med_duration)
print('missing AFTER train:', int(train_clean.isna().sum().sum()), '| test:', int(test_clean.isna().sum().sum()))
print('median duration used:', round(float(med_duration), 2))


### Sec.4 - Feature Engineering
One honest feature: flight_prefix from the flight code recovers airline
signal for the 20% missing-airline rows. Coverage printed below goes to the report.


In [ ]:
# 4. Feature Engineering: flight_prefix recovered from the flight number
# Rationale: 20% of airline labels are missing, but every row has a flight code like 'IX 1070' whose
# prefix is the airline code - this recovers part of the lost signal without touching the target.
for _df in (train_clean, test_clean):
    _df['flight_prefix'] = _df['flight'].fillna('UNK').str.split(' ').str[0]
miss = train['airline'].isna()
cov = float(((train_clean['flight_prefix'] != 'UNK') & miss).sum() / max(1, int(miss.sum())))
print('top prefixes:', train_clean['flight_prefix'].value_counts().head(8).to_dict())
print('share of missing-airline rows with a usable prefix:', round(cov, 3))


### Sec.5 - Data Preprocessing (fit on train only)
ColumnTransformer inside every pipeline: numeric
median-impute + scale; categoricals most-frequent-impute + one-hot (now incl. flight_prefix).


In [ ]:
# 5. Preprocessing pipeline (fit on train only)
# - numeric: duration, days_left -> median impute (duration has NaNs)
# - categoricals -> OneHotEncoder(handle_unknown='ignore')
# - flight (flight number) -> drop or extract airline-code prefix; keep simple: drop
# - id -> keep for submission, drop from X
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

TARGET = 'price'
ID = 'id'
DROP_COLS = [ID, 'flight']  # flight number is identifier-like; optionally engineer prefix
num_features = ['duration', 'days_left']
cat_features = ['airline','source_city','departure_time','stops','arrival_time','destination_city','flight_class','flight_prefix']

numeric_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('scale', StandardScaler()),
])
categorical_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore')),
])
preprocess = ColumnTransformer([
    ('num', numeric_pipe, num_features),
    ('cat', categorical_pipe, cat_features),
])
X = train_clean.drop(columns=[TARGET] + DROP_COLS)
y = train[TARGET]
X_test = test_clean.drop(columns=[ID] + [c for c in DROP_COLS if c in test_clean.columns and c != ID])
print(X.shape, X_test.shape)

### Sec.6 - Model Evaluation: split + reusable functions
80/20 split (seed 42) and RMSE; helpers
regression_report() prints RMSE/R2%/MAPE% for every later section so numbers stay comparable.


In [ ]:
# 6. Train/val split + scoring (RMSE)
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import mean_squared_error
import numpy as np
Xtr, Xva, ytr, yva = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
def rmse(a, b):
    return float(np.sqrt(mean_squared_error(a, b)))
print(Xtr.shape, Xva.shape)

In [ ]:
# 6b. Reusable evaluation functions (used in Sec.7-12)
from sklearn.metrics import r2_score, mean_absolute_percentage_error
def regression_report(name, yt, yp):
    d = {'rmse': rmse(yt, yp), 'r2%': 100 * r2_score(yt, yp),
         'mape%': 100 * mean_absolute_percentage_error(yt, yp)}
    print(f"{name:12s} RMSE {d['rmse']:8.1f} | R2% {d['r2%']:6.2f} | MAPE% {d['mape%']:6.2f}")
    return d


### Sec.7 - Model Family Direct Comparisons
One fit per allowed family at fixed sensible settings -
answers 'which family looks promising before tuning'. Uses Sec.6 helpers + a bar chart.


In [ ]:
# 7. Direct comparison: one fit per model family (fixed sensible settings, no tuning yet)
from sklearn.linear_model import Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
quick = {
    'Ridge(a=1)': Pipeline([('prep', preprocess), ('model', Ridge(alpha=1.0, random_state=RANDOM_STATE))]),
    'Lasso(a=1)': Pipeline([('prep', preprocess), ('model', Lasso(alpha=1.0, max_iter=10000, random_state=RANDOM_STATE))]),
    'Tree(d=8)': Pipeline([('prep', preprocess), ('model', DecisionTreeRegressor(max_depth=8, min_samples_leaf=5, random_state=RANDOM_STATE))]),
}
for _m in quick.values():
    _m.fit(Xtr, ytr)
for _n, _m in quick.items():
    regression_report(_n, yva, _m.predict(Xva))


In [ ]:
# 7b. Bar chart of the direct comparison (shortest bar wins)
import matplotlib.pyplot as plt
_names = ['Ridge(a=1)', 'Lasso(a=1)', 'Tree(d=8)']
_vals = [rmse(yva, quick[n].predict(Xva)) for n in _names]
_o = sorted(range(len(_vals)), key=lambda i: _vals[i])
plt.figure()
plt.bar([_names[i] for i in _o], [_vals[i] for i in _o])
plt.ylabel('val RMSE (lower is better)')
plt.title('Sec.7: one-fit-per-family comparison')
import os; os.makedirs('plots', exist_ok=True)
plt.savefig('plots/sec7_compare.png'); plt.show()


### Sec.8 - Hyper-parameter Tuning: Ridge
GridSearchCV over alpha (5-fold CV, neg-RMSE) + U-curve.
L2 fixes the singular one-hot matrix.


In [ ]:
# 8a. Ridge grid (L2, 5-fold CV)
from sklearn.linear_model import Ridge
pipe_ridge = Pipeline([('prep', preprocess), ('model', Ridge(random_state=RANDOM_STATE))])
grid_ridge = GridSearchCV(pipe_ridge, {'model__alpha': [0.1, 1.0, 10.0, 100.0, 500.0, 1000.0]}, cv=5,
                          scoring='neg_root_mean_squared_error')
grid_ridge.fit(Xtr, ytr)
print('best:', grid_ridge.best_params_)
print('val RMSE:', rmse(yva, grid_ridge.predict(Xva)))

In [ ]:
# Sec.8a plot: Ridge alpha vs CV RMSE
import matplotlib.pyplot as plt
import pandas as pd
r = pd.DataFrame(grid_ridge.cv_results_)
a = pd.to_numeric(r['param_model__alpha'])
s = -r['mean_test_score']
lo = -(r['mean_test_score'] - r['std_test_score'])
hi = -(r['mean_test_score'] + r['std_test_score'])
order = a.sort_values().index
plt.figure()
plt.semilogx(a[order], s[order], marker='o', label='mean CV RMSE')
plt.fill_between(a[order], lo[order], hi[order], alpha=0.2, label='+-1 std')
plt.xlabel('alpha'); plt.ylabel('CV RMSE (lower is better)')
plt.title('Ridge: alpha vs CV RMSE'); plt.legend()
plt.savefig('plots/ridge_alpha.png'); plt.show()
print('best:', grid_ridge.best_params_, '| val RMSE:', round(rmse(yva, grid_ridge.predict(Xva)), 1))


### Sec.8 - Hyper-parameter Tuning: Lasso
Alpha grid + sparsity count: which of the 40+ dummies
survive at exactly zero.


In [ ]:
# 8b. Lasso grid (L1, sparsity)
from sklearn.linear_model import Lasso
pipe_lasso = Pipeline([('prep', preprocess), ('model', Lasso(max_iter=10000, random_state=RANDOM_STATE))])
grid_lasso = GridSearchCV(pipe_lasso, {'model__alpha': [0.01, 0.1, 1.0, 10.0, 100.0]}, cv=5,
                            scoring='neg_root_mean_squared_error')
grid_lasso.fit(Xtr, ytr)
print('best:', grid_lasso.best_params_)
print('val RMSE:', rmse(yva, grid_lasso.predict(Xva)))

In [ ]:
# Sec.8b plot: Lasso alpha vs CV RMSE + sparsity
import matplotlib.pyplot as plt
import pandas as pd
r = pd.DataFrame(grid_lasso.cv_results_)
a = pd.to_numeric(r['param_model__alpha'])
s = -r['mean_test_score']
order = a.sort_values().index
plt.figure()
plt.semilogx(a[order], s[order], marker='o')
plt.xlabel('alpha'); plt.ylabel('CV RMSE (lower is better)')
plt.title('Lasso: alpha vs CV RMSE')
plt.savefig('plots/lasso_alpha.png'); plt.show()
best_coef = grid_lasso.best_estimator_.named_steps['model'].coef_
print('best:', grid_lasso.best_params_, '| nonzero coefs:', int((best_coef != 0).sum()), 'of', len(best_coef))
print('val RMSE:', round(rmse(yva, grid_lasso.predict(Xva)), 1))


### Sec.8 - Hyper-parameter Tuning: Regression Tree
Depth/leaf grid + depth curve, then the
cost-complexity pruning path as a second angle on the same allowed model.


In [ ]:
# 8c. Tree depth grid (DecisionTreeRegressor)
from sklearn.tree import DecisionTreeRegressor
pipe_tree = Pipeline([('prep', preprocess), ('model', DecisionTreeRegressor(random_state=RANDOM_STATE))])
grid_tree = GridSearchCV(pipe_tree,
    {'model__max_depth': [4, 6, 8, 12, None], 'model__min_samples_leaf': [1, 5, 20]},
    cv=5, scoring='neg_root_mean_squared_error')
grid_tree.fit(Xtr, ytr)
print('best:', grid_tree.best_params_)
print('val RMSE:', rmse(yva, grid_tree.predict(Xva)))

In [ ]:
# Sec.8c plot: max_depth vs CV RMSE
import matplotlib.pyplot as plt
import pandas as pd
r = pd.DataFrame(grid_tree.cv_results_)
r['dstr'] = r['param_model__max_depth'].astype(str)
g = r.groupby('dstr')['mean_test_score'].max()
order = sorted(g.index, key=lambda s: (s == 'None', int(s) if s != 'None' else 0))
plt.figure()
plt.plot(order, -g[order].values, marker='o')
plt.xlabel('max_depth'); plt.ylabel('CV RMSE (lower is better)')
plt.title('Tree: depth vs CV RMSE (best min_samples_leaf)')
plt.savefig('plots/tree_depth.png'); plt.show()
print('best:', grid_tree.best_params_, '| val RMSE:', round(rmse(yva, grid_tree.predict(Xva)), 1))


In [ ]:
# 8d. Tree pruning path (ccp_alpha)
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeRegressor
Xtr_t = preprocess.fit_transform(Xtr)
Xva_t = preprocess.transform(Xva)
path = DecisionTreeRegressor(random_state=RANDOM_STATE).cost_complexity_pruning_path(Xtr_t, ytr)
alphas = path.ccp_alphas[::max(1, len(path.ccp_alphas) // 20)]
va_rms = []
for a in alphas:
    t = DecisionTreeRegressor(random_state=RANDOM_STATE, ccp_alpha=a)
    t.fit(Xtr_t, ytr)
    va_rms.append(rmse(yva, t.predict(Xva_t)))
plt.figure()
plt.plot(alphas, va_rms, marker='o')
plt.xlabel('ccp_alpha'); plt.ylabel('val RMSE (lower is better)')
plt.title('Tree: pruning path (ccp_alpha vs val RMSE)')
plt.savefig('plots/tree_prune.png'); plt.show()
best_a = alphas[int(np.argmin(va_rms))]
best_pruned = Pipeline([('prep', preprocess),
    ('model', DecisionTreeRegressor(random_state=RANDOM_STATE, ccp_alpha=best_a))])
best_pruned.fit(Xtr, ytr)
print('best ccp_alpha:', best_a, '| val RMSE:', round(rmse(yva, best_pruned.predict(Xva)), 1))


### Sec.9 - Experiment: Regression on log(price)
Same Ridge model, log1p target, expm1 back.
Tests whether compressing the skewed target helps. Reference says NO (~19.4k vs ~18.3k) - confirm
below and keep raw price.


In [ ]:
# 9. Experiment: Ridge on log1p(price) vs Ridge on raw price (same allowed model)
# Question: does compressing the skewed target (max 323k) help? Fit on log target, invert with expm1.
from sklearn.linear_model import Ridge
_lr = Pipeline([('prep', preprocess), ('model', Ridge(alpha=1.0, random_state=RANDOM_STATE))])
_lr.fit(Xtr, np.log1p(ytr))
_p_log = np.expm1(_lr.predict(Xva)).clip(min=0)
print('--- log-target Ridge ---'); regression_report('Ridge-log', yva, _p_log)
print('--- raw-target Ridge (Sec.8 best) ---'); regression_report('Ridge-raw', yva, grid_ridge.predict(Xva))
print('Conclusion: raw target wins (matches the numpy reference ~19.4k vs ~18.3k) - keep raw price.')


### Sec.10 - OMITTED by design
Ensembles (e.g. Random Forest) are NOT Part-1 models, so the competition
rules ban them. This section is intentionally empty - variety comes from Ridge/Lasso/Tree instead.


### Sec.11 - Model Diagnostics
Residuals of the tuned Tree winner: cloud around 0 and symmetric hist
mean healthy errors; funnel or skew would send us back to Sec.5/8.


In [ ]:
# 11. Model Diagnostics: residuals of the tuned Tree winner on validation
import matplotlib.pyplot as plt
_p = grid_tree.predict(Xva); _r = (yva - _p).astype(float)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].scatter(_p, _r, s=5); ax[0].axhline(0, color='red')
ax[0].set(xlabel='predicted price', ylabel='residual', title='Tuned Tree: residuals')
ax[1].hist(_r, bins=50); ax[1].set(xlabel='residual', title='Tuned Tree: residual histogram')
import os; os.makedirs('plots', exist_ok=True)
plt.tight_layout(); plt.savefig('plots/tree_residuals.png'); plt.show()
regression_report('Tree-tuned', yva, _p)
print('Healthy signs: cloud centered on 0, roughly symmetric hist. Watch for funnel = skew trouble.')


### Sec.12 - Cross-Validation of Shortlisted Models + submission
Honest 5-fold CV on FULL train with
each winner's tuned settings (mean +- std bars), then refit the chosen winner and write submission.csv
for MANUAL upload.


In [ ]:
# 12. Final 5-fold CV of SHORTLISTED models on FULL train + error bars
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
def _strip(_p):
    return {k.split('__', 1)[1]: v for k, v in _p.items()}
_cands = {
    'Ridge': Pipeline([('prep', preprocess), ('model', Ridge(random_state=RANDOM_STATE, **_strip(grid_ridge.best_params_)))]),
    'Lasso': Pipeline([('prep', preprocess), ('model', Lasso(max_iter=10000, random_state=RANDOM_STATE, **_strip(grid_lasso.best_params_)))]),
    'Tree': Pipeline([('prep', preprocess), ('model', DecisionTreeRegressor(random_state=RANDOM_STATE, **_strip(grid_tree.best_params_)))]),
    'Tree-pruned': best_pruned,
}
import matplotlib.pyplot as plt
_names, _means, _stds = [], [], []
for _n, _m in _cands.items():
    _s = -cross_val_score(_m, X, y, cv=5, scoring='neg_root_mean_squared_error')
    _names.append(_n); _means.append(float(_s.mean())); _stds.append(float(_s.std()))
    print(f'{_n:8s} CV RMSE {_s.mean():.1f} +- {_s.std():.1f}')
plt.figure(); plt.errorbar(_names, _means, yerr=_stds, fmt='o', capsize=6)
plt.ylabel('CV RMSE (lower is better)'); plt.title('Sec.12: shortlisted 5-fold CV (mean +- std)')
import os; os.makedirs('plots', exist_ok=True)
plt.savefig('plots/sec12_cv.png'); plt.show()


In [ ]:
# 12b. Refit winner on FULL train -> submission.csv
# Choose best_model manually after comparing above.
best_model = grid_ridge.best_estimator_  # <-- change if another wins
best_model.fit(X, y)
preds = best_model.predict(X_test)
sub = pd.DataFrame({ID: test[ID], TARGET: preds})
assert sub.shape == (len(test), 2), sub.shape
assert list(sub.columns) == list(sample.columns), sample.columns.tolist()
sub.to_csv('submission.csv', index=False)
print(sub.head(), sub.shape)
print('Saved submission.csv - upload MANUALLY to Kaggle. This notebook never submits via API.')